# Задача 10. Graph Convolutional Network



- Найти графовый набор данных для решения задачи предсказания (классификация вершин, обнаружение сообществ и т.д.).
- Использовать несколько слоев GCNConv из библиотеки PyG для построения GCN модели.
- Обучить полученную модель, подобрать гиперпараметры (например, learning rate) на валидационной выборке, и оценить качество предсказания на тестовой выборке.
- (+5 баллов) Также представить самостоятельную реализацию слоя GCNConv, используя матричные операции. Повторить обучение с собственными слоями и сравнить результаты.

In [ ]:
import torch
import torch.nn.functional as F

from torch_geometric.datasets import Planetoid
from torch_geometric.nn import GCNConv
from torch_geometric.utils import add_self_loops, degree

In [ ]:
dataset = Planetoid(root='cora', name='Cora')
data = dataset[0]

Processing...
Done!


In [ ]:
data.num_nodes, data.num_edges, dataset.num_classes

(2708, 10556, 7)

## GCN-модель с слоями GCNConv из PyG

In [ ]:
class GCN(torch.nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super().__init__()
        self.conv1 = GCNConv(input_dim, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, hidden_dim)
        self.conv3 = GCNConv(hidden_dim, output_dim)
        self.dropout = torch.nn.Dropout(0.7)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = self.dropout(x)

        x = self.conv2(x, edge_index)
        x = F.relu(x)
        x = self.dropout(x)

        x = self.conv3(x, edge_index)
        return F.log_softmax(x, dim=1)

In [ ]:
def train():
    model.train()
    optimizer.zero_grad()
    out = model(data.x, data.edge_index)
    loss = F.nll_loss(out[data.train_mask], data.y[data.train_mask])
    loss.backward()
    optimizer.step()
    return loss.item()

def evaluate(mask):
    model.eval()
    with torch.no_grad():
        out = model(data.x, data.edge_index)
        pred = out.argmax(dim=1)
        acc = (pred[mask] == data.y[mask]).sum().item() / mask.sum().item()
    return acc

## Обучение с подбором гиперпараметров

In [ ]:
model = GCN(dataset.num_features, 32, dataset.num_classes)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)

In [ ]:
best_val_acc = 0
best_lr = None

learning_rates = [0.1, 0.01, 0.001]

for lr in learning_rates:
    model = GCN(input_dim=data.num_node_features,
                hidden_dim=16,
                output_dim=dataset.num_classes)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=5e-4)

    current_best_val = 0

    for epoch in range(200):
        train()
        val_acc = evaluate(data.val_mask)

        if val_acc > current_best_val:
            current_best_val = val_acc

        print(f"LR: {lr:.4f}, Epoch: {epoch:03d}, Val Acc: {val_acc:.4f}")

    if current_best_val > best_val_acc:
        best_val_acc = current_best_val
        best_lr = lr

print(f"\nBest Learning Rate: {best_lr}, Validation Accuracy: {best_val_acc:.4f}")

LR: 0.1000, Epoch: 000, Val Acc: 0.1240
LR: 0.1000, Epoch: 001, Val Acc: 0.4020
LR: 0.1000, Epoch: 002, Val Acc: 0.4400
LR: 0.1000, Epoch: 003, Val Acc: 0.5660
LR: 0.1000, Epoch: 004, Val Acc: 0.5820
LR: 0.1000, Epoch: 005, Val Acc: 0.6620
LR: 0.1000, Epoch: 006, Val Acc: 0.6980
LR: 0.1000, Epoch: 007, Val Acc: 0.6620
LR: 0.1000, Epoch: 008, Val Acc: 0.6840
LR: 0.1000, Epoch: 009, Val Acc: 0.6960
LR: 0.1000, Epoch: 010, Val Acc: 0.6900
LR: 0.1000, Epoch: 011, Val Acc: 0.6780
LR: 0.1000, Epoch: 012, Val Acc: 0.7360
LR: 0.1000, Epoch: 013, Val Acc: 0.7560
LR: 0.1000, Epoch: 014, Val Acc: 0.7440
LR: 0.1000, Epoch: 015, Val Acc: 0.7480
LR: 0.1000, Epoch: 016, Val Acc: 0.7180
LR: 0.1000, Epoch: 017, Val Acc: 0.6960
LR: 0.1000, Epoch: 018, Val Acc: 0.7180
LR: 0.1000, Epoch: 019, Val Acc: 0.7340
LR: 0.1000, Epoch: 020, Val Acc: 0.7140
LR: 0.1000, Epoch: 021, Val Acc: 0.7120
LR: 0.1000, Epoch: 022, Val Acc: 0.7200
LR: 0.1000, Epoch: 023, Val Acc: 0.7140
LR: 0.1000, Epoch: 024, Val Acc: 0.7120


## Реализация MyGCNConv и обучение

In [ ]:
class MyGCNConv(torch.nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.lin = torch.nn.Linear(in_channels, out_channels, bias=False)
        self.bias = torch.nn.Parameter(torch.Tensor(out_channels))
        self.reset_parameters()

    def reset_parameters(self):
        self.lin.reset_parameters()
        self.bias.data.zero_()

    def forward(self, x, edge_index):
        num_nodes = x.size(0)
        edge_index, _ = add_self_loops(edge_index, num_nodes=num_nodes)
        row, col = edge_index

        deg = degree(row, num_nodes, dtype=x.dtype)
        deg_inv_sqrt = deg.pow(-0.5)
        deg_inv_sqrt[deg_inv_sqrt == float('inf')] = 0
        norm = deg_inv_sqrt[row] * deg_inv_sqrt[col]

        adj = torch.sparse_coo_tensor(edge_index, norm, (num_nodes, num_nodes))
        x = torch.sparse.mm(adj, x)
        x = self.lin(x)
        x += self.bias
        return x

In [ ]:
class MyGCN(torch.nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super().__init__()
        self.conv1 = MyGCNConv(input_dim, hidden_dim)
        self.conv2 = MyGCNConv(hidden_dim, hidden_dim)
        self.conv3 = MyGCNConv(hidden_dim, output_dim)
        self.dropout = torch.nn.Dropout(0.7)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = self.dropout(x)

        x = self.conv2(x, edge_index)
        x = F.relu(x)
        x = self.dropout(x)

        x = self.conv3(x, edge_index)
        return F.log_softmax(x, dim=1)

In [ ]:
model = MyGCN(dataset.num_features, 16, dataset.num_classes)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)

In [ ]:
best_val_acc = 0
best_lr = None

learning_rates = [0.1, 0.01, 0.001]

for lr in learning_rates:
    model = GCN(input_dim=data.num_node_features,
                hidden_dim=16,
                output_dim=dataset.num_classes)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=5e-4)

    current_best_val = 0

    for epoch in range(200):
        train()
        val_acc = evaluate(data.val_mask)

        if val_acc > current_best_val:
            current_best_val = val_acc

        print(f"LR: {lr:.4f}, Epoch: {epoch:03d}, Val Acc: {val_acc:.4f}")

    if current_best_val > best_val_acc:
        best_val_acc = current_best_val
        best_lr = lr

print(f"\nBest Learning Rate: {best_lr}, Validation Accuracy: {best_val_acc:.4f}")

LR: 0.1000, Epoch: 000, Val Acc: 0.1820
LR: 0.1000, Epoch: 001, Val Acc: 0.5620
LR: 0.1000, Epoch: 002, Val Acc: 0.7000
LR: 0.1000, Epoch: 003, Val Acc: 0.6420
LR: 0.1000, Epoch: 004, Val Acc: 0.7460
LR: 0.1000, Epoch: 005, Val Acc: 0.7660
LR: 0.1000, Epoch: 006, Val Acc: 0.7760
LR: 0.1000, Epoch: 007, Val Acc: 0.7860
LR: 0.1000, Epoch: 008, Val Acc: 0.7780
LR: 0.1000, Epoch: 009, Val Acc: 0.7700
LR: 0.1000, Epoch: 010, Val Acc: 0.7540
LR: 0.1000, Epoch: 011, Val Acc: 0.7560
LR: 0.1000, Epoch: 012, Val Acc: 0.7580
LR: 0.1000, Epoch: 013, Val Acc: 0.7620
LR: 0.1000, Epoch: 014, Val Acc: 0.7580
LR: 0.1000, Epoch: 015, Val Acc: 0.7540
LR: 0.1000, Epoch: 016, Val Acc: 0.7480
LR: 0.1000, Epoch: 017, Val Acc: 0.7420
LR: 0.1000, Epoch: 018, Val Acc: 0.7540
LR: 0.1000, Epoch: 019, Val Acc: 0.7660
LR: 0.1000, Epoch: 020, Val Acc: 0.7700
LR: 0.1000, Epoch: 021, Val Acc: 0.7600
LR: 0.1000, Epoch: 022, Val Acc: 0.7520
LR: 0.1000, Epoch: 023, Val Acc: 0.7400
LR: 0.1000, Epoch: 024, Val Acc: 0.7540


Разница между accuracy после обучением двух моделей совсем небольшая (0.004). Можно утверждать, что самостоятельная реализация оказалась качественной и правильной.